**注意**: 此版本已针对本地Jupyter环境进行优化，无需Google Colab。

# Spacedust: 微生物基因组中保守基因簇的*从头*发现
---

<img src="https://raw.githubusercontent.com/soedinglab/spacedust/master/.github/spacedust.png" height="200" align="right" style="height:200px"/>

Spacedust 是一个模块化工具包，用于基于同源性和基因邻域保守性在多个基因组间识别保守基因簇。Spacedust 采用了 [Foldseek](https://github.com/steineggerlab/foldseek) 快速且敏感的结构比较功能和 [MMseqs2](https://github.com/soedinglab/MMseqs2) 的同源性搜索能力。它引入了一种新方法来聚合基因组对之间的同源性命中集合，并使用凝聚层次聚类算法识别具有保守基因邻域的命中簇。Spacedust 是采用 GPLv3 许可的开源软件，用 C++ 实现，适用于 Linux 和 macOS。该软件旨在在多核环境下高效运行。

**输入要求**

* 每个文件应包含一个基因组。基因组应上传为：
  * `.fna` 文件并开启 `run_prodigal` 选项，或
  * `.faa` 文件，文件头采用 Prodigal 格式
  
**使用说明**

本notebook已适配本地运行环境，您需要：
1. 安装必要的Python依赖包
2. 下载并安装Spacedust软件
3. 准备您的基因组文件
4. 按照下方步骤运行分析

## **环境配置和依赖安装**

在开始使用本notebook之前，请确保您的环境已正确配置。

### **系统要求**
- Linux 或 macOS 系统  
- Python 3.7 或更高版本
- Jupyter Lab 或 Jupyter Notebook
- 至少 2GB 可用磁盘空间

### **Python依赖包安装**

请在运行notebook之前安装以下Python包：

```bash
# 基础科学计算包
pip install numpy pandas matplotlib seaborn

# 下载和解压工具
pip install wget

# Jupyter可视化增强
pip install ipympl

# R接口 (可选，用于高级可视化)
pip install rpy2==3.5.1
```

**或者使用conda环境：**
```bash
# 创建新的conda环境
conda create -n spacedust python=3.9
conda activate spacedust

# 安装依赖
conda install numpy pandas matplotlib seaborn jupyter
conda install -c conda-forge ipympl wget
conda install -c conda-forge rpy2=3.5.1
```

### **R环境配置 (可选)**

如果您希望使用R进行高级基因组上下文可视化，请安装R和相关包：

```r
# 在R中运行
install.packages(c("ggplot2", "gggenes", "RColorBrewer"))
```

### **文件夹结构准备**

在运行notebook的目录中创建以下文件夹：

```bash
mkdir input_genomes     # 存放输入基因组文件
mkdir target_genomes    # 存放目标基因组文件 (如需要)
```

### **注意事项**
- 本notebook已针对本地运行进行优化，无需Google Colab
- 所有软件将自动下载到当前工作目录
- 确保有足够的网络带宽下载Spacedust软件和数据库
- 首次运行时会下载必要的软件包，请耐心等待

In [1]:
# 设置参数
# 任务名称 - 用于输出文件命名
jobname = 'test'

# 输入选项 (当 input_mode 为 all-against-all 时忽略 target_db)
input_mode = "query-target"  # ["query-target", "all-against-all"]
target_db = "KEGG_70"       # ["self-uploaded", "KEGG_70"]
search_mode = "MMseqs2"     # ["MMseqs2"]
run_prodigal = True         # 是否运行 Prodigal 进行基因预测

# 高级选项
max_gene_gap = 3            # 最大基因间隔
num_iterations = 1          # 迭代次数

# 文件路径设置 (请根据实际情况修改)
input_dir = "input_genomes"    # 输入基因组文件夹
target_dir = "target_genomes"  # 目标基因组文件夹 (仅当input_mode为query-target且target_db为self-uploaded时使用)

jobname = "".join(jobname.split())

input_type = 0
if input_mode == "all-against-all":
    input_type = 1

target_type = 0
if target_db != "self-uploaded":
    target_type = 1

search_type = 0
if search_mode == "Foldseek":
    search_type = 1

import os
import shutil

# 检查输入文件夹是否存在
if not os.path.exists(input_dir):
    print(f"请创建输入文件夹: {input_dir}")
    print("并将您的基因组文件放入该文件夹中")
else:
    # 统计输入文件数量
    input_files = [f for f in os.listdir(input_dir) if f.endswith(('.fna', '.faa'))]
    num_queries = len(input_files)
    print(f"找到 {num_queries} 个输入基因组文件")
    
    # 检查all-against-all模式是否有足够的文件
    if input_type == 1 and num_queries <= 1:
        raise ValueError("错误: 'all-against-all' 模式需要多于1个查询文件。")

# 处理目标数据库
if input_type == 0 and target_type == 0:
    if not os.path.exists(target_dir):
        print(f"请创建目标文件夹: {target_dir}")
        print("并将您的目标基因组文件放入该文件夹中")
    else:
        target_files = [f for f in os.listdir(target_dir) if f.endswith(('.fna', '.faa'))]
        print(f"找到 {len(target_files)} 个目标基因组文件")

print("参数设置完成！")
print(f"任务名称: {jobname}")
print(f"输入模式: {input_mode}")
print(f"搜索模式: {search_mode}")
print(f"运行Prodigal: {run_prodigal}")

请创建输入文件夹: input_genomes
并将您的基因组文件放入该文件夹中
参数设置完成！
任务名称: test
输入模式: query-target
搜索模式: MMseqs2
运行Prodigal: True


In [ ]:
# 下载依赖软件和数据库

import subprocess
import os
import wget
import tarfile
import time
import requests
from urllib.error import URLError
import shutil

def download_and_extract(url, extract_path=".", max_retries=3):
    """下载并解压文件，带重试机制"""
    filename = url.split("/")[-1]
    
    if os.path.exists(filename):
        print(f"文件 {filename} 已存在，跳过下载")
        if filename.endswith('.tar.gz'):
            print(f"正在解压 {filename}...")
            with tarfile.open(filename, 'r:gz') as tar:
                tar.extractall(extract_path)
        return True
    
    # 尝试多种下载方法
    for attempt in range(max_retries):
        try:
            print(f"正在下载 {filename} (尝试 {attempt + 1}/{max_retries})...")
            
            # 方法1：使用requests下载（支持断点续传）
            try:
                download_with_requests(url, filename)
                break
            except Exception as e:
                print(f"requests下载失败: {e}")
                
                # 方法2：使用wget下载
                try:
                    wget.download(url, filename)
                    print()
                    break
                except Exception as e2:
                    print(f"wget下载失败: {e2}")
                    
                    if attempt == max_retries - 1:
                        raise Exception(f"所有下载方法都失败了: {e}, {e2}")
                    
                    # 等待后重试
                    print(f"等待10秒后重试...")
                    time.sleep(10)
        
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"下载失败: {e}")
                return False
            time.sleep(5)
    
    # 解压文件
    if filename.endswith('.tar.gz'):
        print(f"正在解压 {filename}...")
        try:
            with tarfile.open(filename, 'r:gz') as tar:
                tar.extractall(extract_path)
            print(f"{filename} 解压完成")
            # 可选：删除压缩包以节省空间
            # os.remove(filename)
            return True
        except Exception as e:
            print(f"解压失败: {e}")
            return False
    
    return True

def download_with_requests(url, filename, chunk_size=8192):
    """使用requests下载文件，支持断点续传"""
    headers = {}
    initial_pos = 0
    
    # 检查是否有部分下载的文件
    if os.path.exists(filename):
        initial_pos = os.path.getsize(filename)
        headers['Range'] = f'bytes={initial_pos}-'
    
    response = requests.get(url, headers=headers, stream=True, timeout=30)
    
    if response.status_code == 416:  # Range not satisfiable，文件已完整
        return
    
    if response.status_code not in [200, 206]:  # 206 for partial content
        response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0)) + initial_pos
    
    with open(filename, 'ab') as f:
        downloaded = initial_pos
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    percent = (downloaded / total_size) * 100
                    print(f"\r下载进度: {percent:.1f}% ({downloaded}/{total_size} bytes)", end='')
        print()

# 下载 Spacedust
if not os.path.exists('SPACEDUST_READY'):
    success = download_and_extract('https://mmseqs.com/spacedust/spacedust-linux-avx2.tar.gz')
    if success:
        with open('SPACEDUST_READY', 'w') as f:
            f.write('')
        print("Spacedust 安装完成")
    else:
        print("Spacedust 下载失败")
else:
    print("Spacedust 已安装")

# 下载 Foldseek (如果需要)
if not os.path.exists('FOLDSEEK_READY') and search_type == 1:
    success = download_and_extract('https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz')
    if success:
        if os.path.exists('foldseek/bin/foldseek'):
            shutil.move('foldseek/bin/foldseek', 'spacedust/bin/')
            shutil.rmtree('foldseek')
        with open('FOLDSEEK_READY', 'w') as f:
            f.write('')
        print("Foldseek 安装完成")
    else:
        print("Foldseek 下载失败")
elif search_type == 1:
    print("Foldseek 已安装")

# 安装 Prodigal (如果需要)
if not os.path.exists('PRODIGAL_READY') and run_prodigal:
    prodigal_url = 'https://github.com/hyattpd/Prodigal/releases/download/v2.6.3/prodigal.linux'
    if not os.path.exists('prodigal'):
        success = download_and_extract(prodigal_url, '.')
        if success:
            # 重命名下载的文件
            if os.path.exists('prodigal.linux'):
                os.rename('prodigal.linux', 'prodigal')
    
    if os.path.exists('prodigal'):
        os.chmod('prodigal', 0o755)
        with open('PRODIGAL_READY', 'w') as f:
            f.write('')
        print("Prodigal 安装完成")
    else:
        print("Prodigal 下载失败")
elif run_prodigal:
    print("Prodigal 已安装")

# 下载数据库 (如果需要)
os.makedirs('database', exist_ok=True)
if not os.path.exists('SPACEDUST_DB_READY') and input_type == 0 and target_type == 1:
    # 修复URL - 使用正确的下载地址
    possible_urls = [
        f'https://wwwuser.gwdg.de/~compbiol/spacedust/{target_db}.tar.gz',
        f'http://wwwuser.gwdg.de/~compbiol/spacedust/{target_db}.tar.gz',
        f'https://ftp.tue.mpg.de/ebio/projects/spacedust/{target_db}.tar.gz'
    ]
    
    print(f"正在下载数据库 {target_db}...")
    os.chdir('database')
    
    success = False
    for url in possible_urls:
        print(f"尝试URL: {url}")
        try:
            success = download_and_extract(url)
            if success:
                break
        except Exception as e:
            print(f"URL {url} 失败: {e}")
            continue
    
    os.chdir('..')
    
    if success:
        with open('SPACEDUST_DB_READY', 'w') as f:
            f.write('')
        print(f"数据库 {target_db} 下载完成")
    else:
        print(f"数据库 {target_db} 下载失败，请手动下载或检查网络连接")
        print("您可以尝试以下方法:")
        print("1. 手动下载 KEGG_70.tar.gz 并放置在 database/ 目录中")
        print("2. 检查网络连接和代理设置")
        print("3. 使用不同的网络环境重试")
        
elif input_type == 0 and target_type == 1:
    print(f"数据库 {target_db} 已存在")

print("依赖软件和数据库准备步骤完成！")

ModuleNotFoundError: No module named 'wget'

In [ ]:
# 运行 Spacedust

import subprocess
import glob

# 设置 Prodigal 路径
prodigal_path = './prodigal' if run_prodigal else 'prodigal'

# 运行 Prodigal 进行基因预测 (如果需要)
if not os.path.exists('PRODIGAL_FAA_READY') and run_prodigal:
    print("开始运行 Prodigal 进行基因预测...")
    
    # 处理输入文件
    fna_files = glob.glob(f"{input_dir}/*.fna")
    if not fna_files:
        raise FileNotFoundError(f"错误: 在 {input_dir} 中未找到 .fna 文件！请确保输入文件扩展名正确！")
    
    for fna_file in fna_files:
        base_name = os.path.splitext(os.path.basename(fna_file))[0]
        faa_file = f"{input_dir}/{base_name}.faa"
        if not os.path.exists(faa_file):
            cmd = [prodigal_path, '-i', fna_file, '-a', faa_file]
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"完成: {base_name}.faa")
    
    # 处理目标文件 (如果需要)
    if input_type == 0 and target_type == 0:
        fna_files = glob.glob(f"{target_dir}/*.fna")
        if not fna_files:
            raise FileNotFoundError(f"错误: 在 {target_dir} 中未找到 .fna 文件！请确保目标文件扩展名正确！")
        
        for fna_file in fna_files:
            base_name = os.path.splitext(os.path.basename(fna_file))[0]
            faa_file = f"{target_dir}/{base_name}.faa"
            if not os.path.exists(faa_file):
                cmd = [prodigal_path, '-i', fna_file, '-a', faa_file]
                subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                print(f"完成: {base_name}.faa")
    
    with open('PRODIGAL_FAA_READY', 'w') as f:
        f.write('')
    print("Prodigal 基因预测完成！")

# 运行 Spacedust
print("开始运行 Spacedust...")

spacedust_bin = './spacedust/bin/spacedust'

if input_type == 1:  # all-against-all 模式
    # 创建数据库
    faa_files = glob.glob(f"{input_dir}/*.faa")
    cmd = [spacedust_bin, 'createsetdb'] + faa_files + [f'database/{jobname}_input', 'tmp', '-v', '0']
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # 运行聚类搜索
    cmd = [spacedust_bin, 'clustersearch', f'database/{jobname}_input', f'database/{jobname}_input', 
           jobname, 'tmp', '--filter-self-match', '--search-mode', str(search_type), 
           '--max-gene-gap', str(max_gene_gap), '-v', '0']
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
else:  # query-target 模式
    # 创建查询数据库
    faa_files = glob.glob(f"{input_dir}/*.faa")
    cmd = [spacedust_bin, 'createsetdb'] + faa_files + [f'database/{jobname}_input', 'tmp', '-v', '0']
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    if target_type == 0:  # 自上传目标
        # 创建目标数据库
        target_faa_files = glob.glob(f"{target_dir}/*.faa")
        cmd = [spacedust_bin, 'createsetdb'] + target_faa_files + [f'database/{jobname}_db', 'tmp', '-v', '0']
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 运行聚类搜索
        cmd = [spacedust_bin, 'clustersearch', f'database/{jobname}_input', f'database/{jobname}_db',
               jobname, 'tmp', '--search-mode', str(search_type), '--max-gene-gap', str(max_gene_gap), '-v', '0']
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
    else:  # 使用预建数据库
        # 运行聚类搜索
        cmd = [spacedust_bin, 'clustersearch', f'database/{jobname}_input', 'database/keggclusterdb',
               jobname, 'tmp', '--search-mode', str(search_type), '--max-gene-gap', str(max_gene_gap), '-v', '0']
        subprocess.run(cmd)

# 后处理结果
print("正在处理结果...")

# 生成前缀ID
cmd = [spacedust_bin, 'prefixid', 'tmp/latest/clusters', f'{jobname}_pref', '--tsv', '-v', '0']
subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 处理输出文件
if os.path.exists(f'{jobname}_pref'):
    # 使用Python替代awk命令处理文件
    import pandas as pd
    
    # 读取pref文件
    pref_df = pd.read_csv(f'{jobname}_pref', sep='\t', header=None, names=['cluid', 'qid', 'tid'])
    
    # 提取qid_tid和cluid
    qid_tid = pref_df[['qid', 'tid']]
    cluid = pref_df[['cluid']]
    
    # 保存临时文件
    qid_tid.to_csv('qid_tid', sep='\t', header=False, index=False)
    cluid.to_csv('cluid', sep='\t', header=False, index=False)
    
    os.remove(f'{jobname}_pref')
    
    # 处理lookup文件
    if os.path.exists(f'database/{jobname}_input.lookup'):
        lookup_df = pd.read_csv(f'database/{jobname}_input.lookup', sep='\t', header=None, names=['id', 'header', 'setid'])
        
        # 创建查询名称映射
        qname_map = dict(zip(lookup_df['id'], lookup_df['header']))
        qid_tid['qname'] = qid_tid['qid'].map(qname_map)
        
        # 处理目标名称映射
        if input_type == 1:
            tname_map = qname_map
        elif target_type == 0:
            target_lookup_df = pd.read_csv(f'database/{jobname}_db.lookup', sep='\t', header=None, names=['id', 'header', 'setid'])
            tname_map = dict(zip(target_lookup_df['id'], target_lookup_df['header']))
        else:
            kegg_lookup_df = pd.read_csv('database/keggclusterdb.lookup', sep='\t', header=None, names=['id', 'header', 'setid'])
            tname_map = dict(zip(kegg_lookup_df['id'], kegg_lookup_df['header']))
        
        qid_tid['tname'] = qid_tid['tid'].map(tname_map)
        
        # 处理setid映射
        qset_map = dict(zip(lookup_df['id'], lookup_df['setid']))
        qid_tid['qsetid'] = qid_tid['qid'].map(qset_map)
        
        if input_type == 1:
            tset_map = qset_map
        elif target_type == 0:
            tset_map = dict(zip(target_lookup_df['id'], target_lookup_df['setid']))
        else:
            tset_map = dict(zip(kegg_lookup_df['id'], kegg_lookup_df['setid']))
        
        qid_tid['tsetid'] = qid_tid['tid'].map(tset_map)
        
        # 分离名称组件
        qid_tid[['qname_base', 'qid_p', 'qid_num', 'qstart', 'qend']] = qid_tid['qname'].str.replace('NZ_', 'NZ.').str.replace('NC_', 'NC.').str.split('_', expand=True)
        qid_tid[['tname_base', 'tid_num', 'tstart', 'tend']] = qid_tid['tname'].str.replace('NZ_', 'NZ.').str.replace('NC_', 'NC.').str.split('_', expand=True)
        
        # 组合最终结果
        result_df = pd.concat([cluid, qid_tid[['qsetid', 'tsetid', 'qid', 'tid']]], axis=1)
        result_df = pd.concat([result_df, qid_tid[['qname_base', 'qid_p', 'qid_num', 'qstart', 'qend', 'tname_base', 'tid_num', 'tstart', 'tend']]], axis=1)
        
        result_df.to_csv(f'{jobname}_plot', sep='\t', header=False, index=False)
    
    # 清理临时文件
    for temp_file in ['qid_tid', 'cluid']:
        if os.path.exists(temp_file):
            os.remove(temp_file)

# 生成输入数据库的前缀ID
cmd = [spacedust_bin, 'prefixid', f'database/{jobname}_input', f'database/{jobname}_input_pref', '--tsv', '-v', '0']
subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Spacedust 运行完成！")

In [ ]:
# 替代方案：手动下载数据库

# 如果自动下载失败，您可以尝试以下方法：

def manual_download_guide():
    """提供手动下载指南"""
    print("=== 手动下载 KEGG_70 数据库指南 ===")
    print()
    print("如果自动下载失败，请按以下步骤手动下载：")
    print()
    print("方法1: 使用curl命令（推荐）")
    print("在终端中运行以下命令：")
    print("cd database")
    print("curl -L -C - -o KEGG_70.tar.gz https://wwwuser.gwdg.de/~compbiol/spacedust/KEGG_70.tar.gz")
    print("# -L 表示跟随重定向，-C - 表示断点续传")
    print()
    print("方法2: 使用wget命令（带重试）")
    print("wget -c -t 0 --timeout=30 https://wwwuser.gwdg.de/~compbiol/spacedust/KEGG_70.tar.gz")
    print("# -c 表示断点续传，-t 0 表示无限重试，--timeout=30 设置超时时间")
    print()
    print("方法3: 分段下载")
    print("如果文件太大，可以考虑分段下载：")
    print("# 下载前半部分")
    print("curl -r 0-500000000 -o KEGG_70.part1 https://wwwuser.gwdg.de/~compbiol/spacedust/KEGG_70.tar.gz")
    print("# 下载后半部分")  
    print("curl -r 500000001- -o KEGG_70.part2 https://wwwuser.gwdg.de/~compbiol/spacedust/KEGG_70.tar.gz")
    print("# 合并文件")
    print("cat KEGG_70.part1 KEGG_70.part2 > KEGG_70.tar.gz")
    print()
    print("方法4: 使用浏览器下载")
    print("1. 在浏览器中访问：https://wwwuser.gwdg.de/~compbiol/spacedust/KEGG_70.tar.gz")
    print("2. 将下载的文件放置在 database/ 目录中")
    print()
    print("下载完成后，解压文件：")
    print("cd database")
    print("tar -xzf KEGG_70.tar.gz")
    print("cd ..")
    print("touch SPACEDUST_DB_READY")  # 标记数据库已准备好
    print()
    print("数据库大小约为：~928MB")

def check_database_status():
    """检查数据库状态"""
    print("=== 数据库状态检查 ===")
    
    database_dir = "database"
    kegg_file = os.path.join(database_dir, "KEGG_70.tar.gz")
    ready_flag = "SPACEDUST_DB_READY"
    
    if os.path.exists(ready_flag):
        print("✓ 数据库标记文件存在")
    else:
        print("✗ 数据库标记文件不存在")
    
    if os.path.exists(kegg_file):
        file_size = os.path.getsize(kegg_file)
        print(f"✓ KEGG_70.tar.gz 存在，大小：{file_size:,} bytes ({file_size/(1024*1024):.1f} MB)")
        
        # 检查文件是否完整（预期大小约为 972659708 bytes）
        expected_size = 972659708
        if abs(file_size - expected_size) < 1000000:  # 允许1MB误差
            print("✓ 文件大小正常")
        else:
            print(f"⚠ 文件大小异常，预期：{expected_size:,} bytes")
    else:
        print("✗ KEGG_70.tar.gz 不存在")
    
    # 检查解压后的文件
    kegg_db_dir = os.path.join(database_dir, "keggclusterdb")
    if os.path.exists(kegg_db_dir):
        print("✓ 数据库目录 keggclusterdb 存在")
    else:
        print("✗ 数据库目录 keggclusterdb 不存在")

def extract_kegg_database():
    """手动解压KEGG数据库"""
    database_dir = "database"
    kegg_file = os.path.join(database_dir, "KEGG_70.tar.gz")
    
    if not os.path.exists(kegg_file):
        print("错误：KEGG_70.tar.gz 文件不存在")
        return False
    
    print("正在解压 KEGG_70.tar.gz...")
    try:
        with tarfile.open(kegg_file, 'r:gz') as tar:
            tar.extractall(database_dir)
        print("✓ 数据库解压完成")
        
        # 创建完成标记
        with open('SPACEDUST_DB_READY', 'w') as f:
            f.write('')
        print("✓ 数据库准备完成标记已创建")
        return True
        
    except Exception as e:
        print(f"解压失败：{e}")
        return False

# 检查当前状态
check_database_status()

# 如果需要，显示手动下载指南
if not os.path.exists('SPACEDUST_DB_READY'):
    print()
    manual_download_guide()

In [ ]:
# 显示聚类匹配结果
from pathlib import Path
from IPython.display import HTML
from IPython.core.display import display

html = '''
<style>
tbody tr.head {
  font-weight: bold;
  background-color: #f2f2f2;
}
</style>
<table style='text-align:left'>
  <thead>
    <tr>
      <th>聚类匹配ID</th>
      <th>查询序列</th>
      <th>目标序列</th>
      <th>聚类匹配P值</th>
      <th colspan="8">命中数</th>
    </tr>
    <tr>
      <th colspan='2'>查询ID</th>
      <th>目标ID</th>
      <th>序列相似度</th>
      <th>E值</th>
      <th>查询起始</th>
      <th>查询结束</th>
      <th>查询长度</th>
      <th>目标起始</th>
      <th>目标结束</th>
      <th>目标长度</th>
      <th>比对Cigar</th>
    </tr>
  </thead>
<tbody>
'''

# 检查结果文件是否存在
result_file = Path(jobname)
if result_file.exists():
    for line in result_file.read_text().split('\n'):
        if len(line) == 0:
            continue
        if line[0] == '#':
            cols = line.split('\t')
            if len(cols) == 6:
                html += "<tr class='head'>"
                html += "<td>" + cols[0][1:] + "</td>"
                html += "<td>" + cols[1] + "</td>"
                html += "<td>" + cols[2] + "</td>"
                html += "<td>" + cols[4] + "</td>"
                html += "<td colspan='8'>" + cols[5] + "</td>"
                html += "</tr>\n"
        elif line[0] == '>':
            cols = line.split('\t')
            if len(cols) == 12:
                html += "<tr>"
                html += "<td colspan='2'>" + cols[0][1:] + "</td>"
                html += "<td>" + cols[1] + "</td>"
                html += "<td>" + cols[3] + "</td>"
                html += "<td>" + cols[4] + "</td>"
                html += "<td>" + cols[5] + "</td>"
                html += "<td>" + cols[6] + "</td>"
                html += "<td>" + cols[7] + "</td>"
                html += "<td>" + cols[8] + "</td>"
                html += "<td>" + cols[9] + "</td>"
                html += "<td>" + cols[10] + "</td>"
                html += "<td>" + cols[11] + "</td>"
            html += "</tr>\n"
    html += '''
    </tbody>
    </table>
    '''
    display(HTML(html))
    print(f"结果已保存到文件: {jobname}")
else:
    print(f"未找到结果文件: {jobname}")
    print("请先运行上述步骤生成结果")

In [ ]:
# 结果文件说明

import os

print("=== Spacedust 分析结果 ===")
print(f"主要结果文件: {jobname}")
print(f"可视化数据文件: {jobname}_plot")

# 列出所有生成的文件
result_files = []
for file_pattern in [jobname, f"{jobname}_plot", "database", "tmp"]:
    if os.path.exists(file_pattern):
        if os.path.isfile(file_pattern):
            result_files.append(file_pattern)
        elif os.path.isdir(file_pattern):
            result_files.append(f"{file_pattern}/ (目录)")

if result_files:
    print("\n生成的文件和目录:")
    for file in result_files:
        print(f"  - {file}")
else:
    print("未找到结果文件，请检查运行过程是否有错误。")

print(f"\n所有结果文件保存在当前工作目录中。")
print("您可以使用这些文件进行进一步分析或与他人共享。")

# **结果可视化**

In [ ]:
# 准备可视化数据

# 安装必要的可视化包
import subprocess
import sys

def install_package(package):
    """安装Python包"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"{package} 安装成功")
    except:
        print(f"{package} 安装失败，请手动安装")

# 检查并安装必要的包
required_packages = ["ipympl", "rpy2==3.5.1"]
for package in required_packages:
    try:
        if package.startswith("rpy2"):
            import rpy2
        elif package.startswith("ipympl"):
            import ipympl
    except ImportError:
        install_package(package)

import os
import numpy as np
import pandas as pd
import seaborn as sns

# 检查结果文件是否存在
if not os.path.exists(f"{jobname}_plot"):
    print(f"错误: 未找到文件 {jobname}_plot")
    print("请先运行 Spacedust 分析步骤")
else:
    # 读取匹配结果
    matchhit = pd.read_csv(f"{jobname}_plot", sep="\t", 
                          names=['cluid','qsetid', 'tsetid', 'qseqid','tseqid',
                                'qname', 'qid_p','qid','qstart','qend', 
                                'tname','tid','tstart','tend'])
    
    # 读取lookup文件
    lookup = pd.read_csv(f"database/{jobname}_input.lookup", sep="\t", 
                        names=['seqid','header','setid'])
    
    # 读取序列信息
    all_seq = pd.read_csv(f"database/{jobname}_input_pref", sep="\t", 
                         names=['id','seq'], header=None, 
                         dtype={'id': int, 'seq': str})
    
    print("可视化数据准备完成！")
    print(f"匹配结果记录数: {len(matchhit)}")
    print(f"查询序列数: {len(lookup)}")
    print(f"总序列数: {len(all_seq)}")

In [ ]:
# 选择查询基因组

# 检查源文件是否存在
source_file = f'database/{jobname}_input.source'
if not os.path.exists(source_file):
    print(f"警告: 未找到源文件 {source_file}")
    print("将使用默认设置: query_genome = 0")
    query_genome = 0
else:
    # 读取基因组信息
    df = pd.read_csv(source_file, sep="\t", 
                    names=['query_genome_id','query_genome_name'],
                    dtype={'query_genome_id': int, 'query_genome_name': str})
    
    print("可用的查询基因组:")
    for idx, row in df.iterrows():
        print(f"  {row['query_genome_id']}: {row['query_genome_name']}")
    
    if len(df) == 1:
        query_genome = 0
        print(f"\n自动选择唯一基因组: {df.iloc[0]['query_genome_name']}")
    else:
        # 手动选择基因组 (对于多个基因组的情况)
        print(f"\n请修改下面的 query_genome 值来选择基因组 (0-{len(df)-1}):")
        query_genome = 0  # 默认选择第一个
        print(f"当前选择: {df.iloc[query_genome]['query_genome_name']}")

# 计算匹配统计
if 'matchhit' in locals():
    # 创建匹配计数数组
    matchhit_temp = matchhit[matchhit['qsetid'] == query_genome]
    if len(matchhit_temp) > 0:
        qid = matchhit_temp.drop_duplicates(['qid','tsetid'], keep='last')['qid'].to_numpy()
        matchhit_array = np.zeros(qid.max()+1, dtype=int)
        for i in qid:
            matchhit_array[i] += 1
        
        # 创建保守性矩阵计数数组
        qid = matchhit.sort_values(['cluid', 'qid'])['qid'].to_numpy()
        tid = matchhit.sort_values(['cluid', 'qid'])['tid'].to_numpy()
        cluid = matchhit.sort_values(['cluid', 'qid'])['cluid'].to_numpy()
        matchpair_array = np.zeros(qid.max(), dtype=int)
        
        for i in np.arange(len(qid)-1):
            if(cluid[i] == cluid[i+1]):
                if(qid[i] == qid[i+1]-1):
                    matchpair_array[qid[i]] += 1
                else:
                    if(abs(qid[i+1]-qid[i]) == abs(tid[i+1]-tid[i])):
                        for x in np.arange(qid[i], qid[i+1]):
                            matchpair_array[x] += 1
        
        count = np.zeros(matchhit_array.max()+1)
        for i in matchhit_array.tolist():
            count[i] += 1
        
        print(f"已为基因组 {query_genome} 计算匹配统计")
    else:
        print(f"警告: 基因组 {query_genome} 没有找到匹配结果")

# 处理lookup信息
if 'lookup' in locals():
    lookup_temp = lookup[lookup['setid'] == query_genome]
    if len(lookup_temp) > 0:
        lookup_temp['idx'] = lookup_temp['header'].str.split('_').str[-3].astype(int)
        lookup_temp['qstart'] = lookup_temp['header'].str.split('_').str[-2].astype(int)
        lookup_temp['qend'] = lookup_temp['header'].str.split('_').str[-1].astype(int)
        print(f"处理了 {len(lookup_temp)} 个基因的lookup信息")
    else:
        print(f"警告: 基因组 {query_genome} 没有找到lookup信息")

print("基因组选择和数据处理完成！")

In [ ]:
# 聚类匹配热图/柱状图

# 可视化参数设置
Zoom = False  # 是否缩放显示
lower_bound = 1   # 缩放下界 (当Zoom=True时生效)
upper_bound = 100 # 缩放上界 (当Zoom=True时生效)

# 确保下界小于上界
if lower_bound >= upper_bound:
    raise ValueError("下界必须小于上界")

# 导入可视化库
import matplotlib.pyplot as plt
import numpy as np
import math
import matplotlib.cm as cm
from matplotlib.widgets import Slider
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable

# 检查是否有匹配数据
if 'matchhit' not in locals() or 'matchhit_array' not in locals():
    print("错误: 请先运行数据准备和基因组选择步骤")
else:
    # 设置matplotlib后端
    plt.switch_backend('agg')  # 对于非交互式环境
    
    # 获取最大蛋白质ID和基因组ID
    max_protein_id = np.max(matchhit['qid'])
    max_genome_id = np.max(matchhit['tsetid'])
    
    # 创建存在/缺失矩阵
    matrix = np.zeros((max_genome_id + 1, max_protein_id + 1))
    matrix[query_genome, :] = 1
    
    # 填充蛋白质命中数据
    for _, row in matchhit.iterrows():
        matrix[row['tsetid'], row['qid']] = 1
    
    # 创建链方向图矩阵
    if 'lookup_temp' in locals() and len(lookup_temp) > 0:
        strand_plot = np.zeros(len(lookup_temp['idx']), dtype=int)
        # 标记正负链
        for _, row in lookup_temp.iterrows():
            if (row['qstart'] < row['qend']):
                strand_plot[row['idx']] = 1
    else:
        strand_plot = np.array([])
    
    # 创建图形
    fig, axes = plt.subplots(nrows=3, sharex=True, figsize=(12, 10), 
                            gridspec_kw={'height_ratios': [4, 0.1, 1]})
    ax1, ax3, ax2 = axes
    
    # 创建热图
    heatmap = ax1.imshow(matrix, cmap='Blues', interpolation='none', aspect='auto')
    ax1.set_ylabel('基因组ID')
    ax1.set_title('蛋白质存在/缺失热图')
    
    # 设置y轴标签
    y_labels = range(0, max_genome_id + 1)
    ax1.set_yticks(range(0, max_genome_id + 1))
    ax1.set_yticklabels(y_labels)
    
    # 设置x轴刻度
    ax1.xaxis.set_major_locator(ticker.AutoLocator())
    ax1.yaxis.set_major_locator(ticker.AutoLocator())
    
    # 添加柱状图
    if 'matchpair_array' in locals() and 'matchhit_array' in locals():
        ax2.bar(np.arange(len(matchpair_array)), matchpair_array, width=1, 
                align='edge', edgecolor='black', color='lightpink', 
                label='基因邻域保守性')
        ax2.bar(np.arange(len(matchhit_array)), matchhit_array, width=0.5, 
                align='center', label='命中计数')
        ax2.set_ylabel('命中计数')
        ax2.set_xlabel('查询蛋白质位置索引')
        ax2.legend()
        ax2.set_yscale('log')
    
    # 设置x轴范围
    if Zoom:
        ax2.set_xlim(lower_bound-0.5, upper_bound + 0.5)
    else:
        ax2.set_xlim(-0.5, max_protein_id+1 - 0.5)
    
    ax2.xaxis.set_major_locator(ticker.AutoLocator())
    
    # 添加链方向图
    if len(strand_plot) > 0:
        cmap_binary = cm.binary
        ax3.imshow(strand_plot.reshape(1, -1), cmap=cmap_binary, aspect='auto')
        ax3.set_ylabel('链方向')
        ax3.yaxis.set_ticks([])
    
    # 保存图形
    plt.tight_layout()
    plt.savefig('spacedust_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig('spacedust_heatmap.pdf', bbox_inches='tight')
    
    print("热图已保存为 'spacedust_heatmap.png' 和 'spacedust_heatmap.pdf'")
    
    # 显示图形 (如果支持的话)
    try:
        plt.show()
    except:
        print("无法显示图形，但已保存到文件中")

In [ ]:
# 聚类匹配柱状图 (简化版)

# 检查是否有匹配数据
if 'matchhit_array' not in locals() or 'matchpair_array' not in locals():
    print("错误: 请先运行数据准备步骤")
else:
    # 创建简化的静态柱状图
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('white')
    
    x = np.arange(len(matchhit_array))
    y1 = matchhit_array
    y2 = matchpair_array
    
    # 显示范围设置
    N = min(100, len(x))  # 最多显示100个基因
    pos = 0  # 起始位置
    
    if pos + N > len(x):
        n = len(x) - pos
    else:
        n = N
    
    X = x[pos:pos+n]
    Y = y1[pos:pos+n]
    Y2 = y2[pos:pos+n]
    
    # 绘制柱状图
    ax.bar(X, Y2, width=1, align='edge', edgecolor='black', color='lightgrey', 
           label='基因邻域保守性')
    ax.bar(X, Y, width=0.5, align='edge', edgecolor='black', color='steelblue',
           label='命中计数')
    
    ax.set_xlabel('基因位置')
    ax.set_ylabel('计数')
    ax.set_title(f'基因 {pos+1}-{pos+n} 的匹配统计')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 保存图形
    plt.tight_layout()
    plt.savefig('spacedust_barplot.png', dpi=300, bbox_inches='tight')
    plt.savefig('spacedust_barplot.pdf', bbox_inches='tight')
    
    print("柱状图已保存为 'spacedust_barplot.png' 和 'spacedust_barplot.pdf'")
    
    # 显示图形
    try:
        plt.show()
    except:
        print("无法显示图形，但已保存到文件中")
    
    # 如果您想查看不同区域的基因，可以修改上面的 pos 值重新运行此cell
    print(f"\n提示: 要查看不同区域的基因，请修改 pos 值 (当前: {pos})，范围: 0-{len(x)-N}")
    print(f"总共有 {len(x)} 个基因位置")

In [ ]:
# 选择感兴趣的蛋白质，提取包含这些蛋白质的所有聚类匹配

# 查询蛋白质ID设置 (单个整数或范围，例如 '1' 或 '1-10')
query_protein_id_input = '1'  # 可以修改此值来选择不同的蛋白质

def parse_input(input_str):
    """解析输入字符串获取蛋白质ID列表"""
    parts = input_str.split('-')
    if len(parts) == 1:
        # 单个整数
        return [int(parts[0])]
    elif len(parts) == 2:
        # 范围
        return list(range(int(parts[0]), int(parts[1]) + 1))
    else:
        raise ValueError("输入格式无效。请输入单个整数或范围。")

# 解析输入
try:
    query_protein_id = parse_input(query_protein_id_input)
    print(f"选择的蛋白质ID: {query_protein_id}")
except ValueError as e:
    print(f"错误: {e}")
    query_protein_id = [1]  # 默认值

# 检查是否有匹配数据
if 'matchhit' not in locals():
    print("错误: 请先运行数据准备步骤")
else:
    # 过滤聚类ID，仅包含具有所有查询蛋白质ID的聚类
    clusterid = matchhit.loc[matchhit['qid'].isin(query_protein_id), 'cluid'].unique().tolist()
    
    # 检查每个聚类中是否包含所有查询蛋白质ID
    filtered_clusterid = []
    for cluster in clusterid:
        cluster_proteins = set(matchhit[matchhit['cluid'] == cluster]['qid'].unique())
        if set(query_protein_id).issubset(cluster_proteins):
            filtered_clusterid.append(cluster)
    
    print(f"找到 {len(filtered_clusterid)} 个包含所选蛋白质的聚类")
    
    # 提取相关数据
    appended_data = pd.DataFrame()
    for cluster_id in filtered_clusterid:
        cluster_data = matchhit[matchhit['cluid'] == cluster_id]
        appended_data = pd.concat([appended_data, cluster_data])
    
    # 过滤重复的qseqid
    if len(appended_data) > 0:
        appended_data = appended_data[appended_data['qseqid'].map(appended_data['qseqid'].value_counts()) > 1]
        print(f"过滤后的匹配记录数: {len(appended_data)}")
        
        if len(appended_data) > 0:
            print("数据准备完成，可以进行基因组上下文可视化")
        else:
            print("警告: 过滤后没有剩余数据")
    else:
        print("警告: 未找到匹配的聚类数据")
        appended_data = pd.DataFrame()  # 空DataFrame

In [ ]:
# 转换为R数据框进行基因组上下文可视化

# 检查R环境和必要包
try:
    import rpy2
    from rpy2.robjects import pandas2ri
    print("R环境已就绪")
except ImportError:
    print("警告: rpy2未安装或配置不正确")
    print("请运行: pip install rpy2")
    print("或使用conda: conda install rpy2")

# 检查是否有数据
if 'appended_data' not in locals() or len(appended_data) == 0:
    print("错误: 没有可用的聚类数据，请先运行蛋白质选择步骤")
else:
    # 设置预定义的查询序列ID
    predefined_qseqid = query_protein_id[0]
    
    def invert_sign(group):
        """反转符号以标准化方向"""
        matching_rows = group[group['qseqid'] == predefined_qseqid]
        
        if not matching_rows.empty:
            predefined_row = matching_rows.iloc[0]
            q_direction = np.sign(predefined_row['qstart'] - predefined_row['qend'])
            t_direction = np.sign(predefined_row['tstart'] - predefined_row['tend'])
            
            if q_direction != t_direction:
                group['tstart'], group['tend'] = -group['tstart'].values, -group['tend'].values
        
        return group
    
    # 按聚类分组并应用方向标准化
    appended_data_g = appended_data.groupby('cluid', group_keys=False).apply(invert_sign)
    
    center = str(query_protein_id[0])
    
    # 过滤掉查询和目标相同的记录
    appended_data_g = appended_data_g[appended_data_g['tname'] != appended_data_g['qname']]
    
    if len(appended_data_g) == 0:
        print("警告: 过滤后没有剩余数据用于可视化")
    else:
        # 创建基因数据框
        gggene_df = appended_data_g.groupby(by="qid", as_index=False).first()[['qname','qid','qstart','qend']]
        
        # 添加目标基因信息
        target_genes = appended_data_g[['tname','qid','tstart','tend']].rename(
            columns={"tname": "qname", "tstart": "qstart", "tend": "qend"})
        
        gggene_df = pd.concat([gggene_df, target_genes]).reset_index(drop=True)
        
        print(f"准备用于R可视化的基因数据: {len(gggene_df)} 个基因")
        print(f"涉及基因组数量: {gggene_df['qname'].nunique()}")
        
        # 如果R环境可用，转换数据
        try:
            pandas2ri.activate()
            r_dataframe = pandas2ri.py2rpy(gggene_df)
            print("数据已成功转换为R数据框")
        except:
            print("R数据转换失败，但Python数据框已准备好")
            print("您可以将数据导出并在R中手动处理")

In [ ]:
# 使用R进行基因组上下文可视化

# 检查R环境
try:
    # 加载R扩展
    get_ipython().run_line_magic('load_ext', 'rpy2.ipython')
    
    # R代码块
    get_ipython().run_cell_magic('R', '-i gggene_df,center', '''
    # 安装并加载必要的R包
    if (!require("gggenes", quietly = TRUE)) {
        install.packages("gggenes", quiet = TRUE)
        library(gggenes)
    }
    if (!require("ggplot2", quietly = TRUE)) {
        install.packages("ggplot2", quiet = TRUE)
        library(ggplot2)
    }
    if (!require("RColorBrewer", quietly = TRUE)) {
        install.packages("RColorBrewer", quiet = TRUE)
        library(RColorBrewer)
    }
    
    # 检查数据
    if (nrow(gggene_df) == 0) {
        cat("错误: 没有数据用于可视化\\n")
    } else {
        # 设置颜色
        nb.cols <- length(unique(gggene_df$qid))
        mycolors <- colorRampPalette(brewer.pal(min(8, nb.cols), "Set3"))(nb.cols)
        
        # 创建对齐虚拟数据
        dummies <- make_alignment_dummies(
            gggene_df,
            aes(xmin = qstart, xmax = qend, y = qname, id = qid),
            on = center
        )
        
        # 创建基因箭头图
        p <- ggplot(gggene_df, aes(xmin = qstart, xmax = qend, y = qname, fill = factor(qid))) +
            geom_gene_arrow() +
            geom_blank(data = dummies) +
            facet_wrap(~ qname, scales = "free", ncol = 1) +
            scale_fill_manual(values = mycolors) +
            theme_genes() +
            labs(fill = "基因", y = "基因组", x = "基因组位置") +
            theme(axis.text.y = element_text(size = 8))
        
        # 计算图形尺寸
        width <- max(6, 1.5 * length(unique(gggene_df$qid)))
        height <- max(4, 0.5 * length(unique(gggene_df$qname)))
        
        # 保存图形
        ggsave("spacedust_gene_context.png", plot = p, width = width, height = height, dpi = 300)
        ggsave("spacedust_gene_context.pdf", plot = p, width = width, height = height)
        
        # 显示图形
        print(p)
        
        cat("基因组上下文图已保存为 spacedust_gene_context.png 和 spacedust_gene_context.pdf\\n")
    }
    ''')
    
except Exception as e:
    print(f"R可视化失败: {e}")
    print("请检查:")
    print("1. 是否安装了rpy2: pip install rpy2")
    print("2. 是否安装了R及相关包")
    print("3. Python和R之间的接口是否正常")
    
    # 提供Python替代方案
    print("\n使用Python替代可视化:")
    if 'gggene_df' in locals() and len(gggene_df) > 0:
        # 简单的Python可视化
        fig, ax = plt.subplots(figsize=(12, max(4, len(gggene_df['qname'].unique()) * 0.5)))
        
        # 为每个基因组绘制基因
        y_positions = {name: i for i, name in enumerate(gggene_df['qname'].unique())}
        colors = plt.cm.Set3(np.linspace(0, 1, len(gggene_df['qid'].unique())))
        color_map = {qid: colors[i] for i, qid in enumerate(sorted(gggene_df['qid'].unique()))}
        
        for _, row in gggene_df.iterrows():
            y = y_positions[row['qname']]
            x_start, x_end = row['qstart'], row['qend']
            color = color_map[row['qid']]
            
            # 绘制基因箭头
            if x_start < x_end:  # 正向
                ax.arrow(x_start, y, x_end - x_start, 0, head_width=0.1, 
                        head_length=(x_end - x_start) * 0.1, fc=color, ec='black')
            else:  # 反向
                ax.arrow(x_start, y, x_end - x_start, 0, head_width=0.1, 
                        head_length=(x_start - x_end) * 0.1, fc=color, ec='black')
        
        ax.set_yticks(list(y_positions.values()))
        ax.set_yticklabels(list(y_positions.keys()))
        ax.set_xlabel('基因组位置')
        ax.set_ylabel('基因组')
        ax.set_title('基因组上下文 (简化版)')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('spacedust_gene_context_python.png', dpi=300, bbox_inches='tight')
        print("Python版本的基因组上下文图已保存为 spacedust_gene_context_python.png")
        plt.show()
    else:
        print("没有可用的基因数据")

In [ ]:
# 查看基因组上下文并保存图像

import os
from IPython.display import Image, display

# 检查生成的图像文件
image_files = [
    ("spacedust_gene_context.png", "R版本基因组上下文图"),
    ("spacedust_gene_context_python.png", "Python版本基因组上下文图"),
    ("spacedust_heatmap.png", "聚类匹配热图"),
    ("spacedust_barplot.png", "聚类匹配柱状图")
]

print("=== 生成的可视化文件 ===")
for filename, description in image_files:
    if os.path.exists(filename):
        print(f"✓ {filename} - {description}")
        try:
            # 尝试显示图像
            display(Image(filename=filename, width=600))
        except:
            print(f"  无法显示图像，但文件已保存")
    else:
        print(f"✗ {filename} - 未生成")

print("\n=== 所有输出文件 ===")
output_files = [
    jobname,
    f"{jobname}_plot",
    "spacedust_heatmap.png",
    "spacedust_heatmap.pdf", 
    "spacedust_barplot.png",
    "spacedust_barplot.pdf",
    "spacedust_gene_context.png",
    "spacedust_gene_context.pdf",
    "spacedust_gene_context_python.png"
]

for filename in output_files:
    if os.path.exists(filename):
        if os.path.isfile(filename):
            size = os.path.getsize(filename)
            print(f"✓ {filename} ({size} bytes)")
        elif os.path.isdir(filename):
            print(f"✓ {filename}/ (目录)")

print(f"\n所有结果文件保存在当前工作目录中:")
print(f"当前目录: {os.getcwd()}")

# 提供使用建议
print("\n=== 使用建议 ===")
print("1. 查看主要结果文件:", jobname)
print("2. 分析可视化图片了解基因簇的分布和保守性")
print("3. 根据需要调整参数重新运行分析")
print("4. 所有PDF文件适合用于发表或报告")

# **使用说明**

## **快速开始**
1. 安装必要的Python依赖包 (见下方环境配置部分)
2. 设置参数并准备基因组文件
3. 按顺序运行各个代码块
4. 当前运行的步骤会在代码块左侧显示运行状态

## **结果文件内容**

1. 制表符分隔文本文件 (`.tsv`) 包含所有报告的聚类匹配结果
2. 可视化图像文件 (`.png`, `.pdf`) 用于结果展示
3. (如适用) Prodigal预测的蛋白质序列 (`.faa`)

## **环境配置要求**

### **必需的Python包**
```bash
pip install numpy pandas matplotlib seaborn ipympl wget
pip install rpy2==3.5.1  # 用于R可视化 (可选)
```

### **系统要求**
- Linux 或 macOS 系统
- Python 3.7+
- 足够的磁盘空间存储基因组文件和数据库

### **输入文件准备**
- 创建 `input_genomes` 文件夹
- 将查询基因组文件放入其中 (`.fna` 或 `.faa` 格式)
- 如使用自定义目标数据库，创建 `target_genomes` 文件夹

## **故障排除**
- 如果遇到依赖包安装问题，请使用 conda 环境
- 检查输入基因组文件是否已解压缩
- 确保有足够的磁盘空间用于临时文件
- R可视化部分为可选功能，失败不影响主要分析

## **性能限制**
- 由于计算资源限制，默认仅支持MMseqs2同源性搜索
- 对于大规模数据集，建议在高性能计算环境中运行
- 内存使用量取决于基因组数量和大小

## **问题反馈**
- 如遇到程序错误，请访问 https://github.com/soedinglab/spacedust/issues 报告问题
- 本notebook的改进建议也欢迎反馈

## **引用信息**
如果使用Spacedust进行研究，请引用相关论文。详细信息请访问官方GitHub页面。